# Reproducing Iacopo Bertelli's Counter-Intuitive Flows Analysis Articles

Iacopo Bertelli's [Counterintuitive Flows in 2023 series](https://www.linkedin.com/pulse/counterintuitive-flows-2023-part-2-3-iacopo-bertelli-baonf) is a masterpiece of analysis to find three specific hours when: 

- European day-ahead prices converged almost everywhere
- the most bidding-zone borders were simultaneously congested
- the most borders had counter-intuitive flows, that is power flowing the "wrong" way relative to the price signal

This notebook builds on top of the Voltpath API and reproduces the second article's three 2023 moments, then builds a leaderboard of the borders where counter-intuitive (CI) flows show up most.


**Before you run this:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key (find it at [voltpath.ai/settings](https://voltpath.ai/settings) once logged in), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY` (or leave `.env` empty and you'll be prompted).

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai) &ndash; if you're using an AI coding agent with the Context7 MCP server, it can pull these docs directly). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
from common import (
    FLOW_COLORS,
    border_count_price_chart,
    convergence_summary,
    fetch_flows,
    fetch_flows_range,
    flow_type_summary,
    hourly_border_counts,
    peak,
    save_chart,
    top_n_bar,
)

## Fetching flow data

`GET /api/v1/flows` joins physical flow with day-ahead prices on both sides of a border and classifies each observation into `Convergence`, `Congestion`, `Counter-intuitive`, or `Outage` &ndash; see [Flow Types](https://docs.voltpath.ai/concepts/flow-types). 

The API has no limit on how far back `datetime_from`/`datetime_to` can reach; it just downgrades resolution automatically as the range widens to keep responses a manageable size. Pass `resolution="native"` explicitly to keep 15-minute granularity on short windows.

`fetch_flows` (in [`articles/common.py`](../common.py), shared across all three flow-analysis notebooks) wraps that call, caches the result to disk, and adds a few columns every analysis below needs.

## Reproducing Bertelli's three 2023 moments

[Counterintuitive Flows in 2023, Part 2 of 3](https://www.linkedin.com/pulse/counterintuitive-flows-2023-part-2-3-iacopo-bertelli-baonf) called out three specific hours, we'll look at each next.

### Full price convergence &ndash; 2023-12-05, 19:00 UTC

EUPHEMIA at its best: [full price convergence on 2023-12-05](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2023-12-05&dateTo=2023-12-06). 

VoltPath displays UTC, so 19:00 UTC is 20:00 CET; the dashboard's playback control defaults to midnight, so drag it forward to that hour to see this moment.

<img src="media/convergence-2023-12-05-dashboard.png" alt="VoltPath dashboard showing full price convergence on 2023-12-05" width="800">

In [ ]:
convergence_day = fetch_flows("2023-12-05", "2023-12-06", resolution="native")
peak_row = peak(hourly_border_counts(convergence_day, "Convergence"))
print(f"Peak: {peak_row['n_borders']} borders converged at {peak_row['datetime']}")

fig = border_count_price_chart(
    convergence_day,
    "Convergence",
    "Converged borders vs. average day-ahead price, 2023-12-05 (UTC)",
)
save_chart(fig, "convergence-2023-12-05")
fig.show()

convergence_summary(convergence_day).head(10)

### Peak simultaneous congestion &ndash; 2023-02-22, 13:00 UTC

Converged prices are the ideal state, congested bidding zones are the typical: [congestion on top of congestion on 2023-02-22](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2023-02-22&dateTo=2023-02-23).

Again, drag the playback control to 13:00 UTC / 14:00 CET, since it defaults to midnight.

<img src="media/congestion-peak-2023-02-22-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2023-02-22" width="800">

This is the repro chart only; the congestion leaderboard (which borders are congested most, and at what implied cost) lives in the [congestion-analysis](../congestion-analysis/congestion_analysis.ipynb) notebook.

In [ ]:
congestion_day = fetch_flows("2023-02-22", "2023-02-23", resolution="native")
peak_row = peak(hourly_border_counts(congestion_day, "Congestion"))
print(f"Peak: {peak_row['n_borders']} borders congested at {peak_row['datetime']}")

fig = border_count_price_chart(
    congestion_day,
    "Congestion",
    "Congested borders vs. average day-ahead price, 2023-02-22 (UTC)",
    peak_vline="2023-02-22 13:00:00",
    peak_label="Bertelli's reported peak",
)
save_chart(fig, "congestion-peak-2023-02-22")
fig.show()

### Peak counter-intuitive flow &ndash; 2023-07-09, 09:00 UTC

Perhaps the most exciting of all snapshots: Bertelli reported 19 borders with counter-intuitive flow at 11:00 CET (09:00 UTC). 

[Europe turns to a web of CI flows on 2023-07-09](https://voltpath.ai/dashboard?dateFrom=2023-07-09&dateTo=2023-07-10&viewMode=flows-prices) &ndash; drag playback to 09:00 UTC / 11:00 CET.

<img src="media/ci-peak-2023-07-09-dashboard.png" alt="VoltPath dashboard showing peak counter-intuitive flow on 2023-07-09" width="800">

In [ ]:
ci_day = fetch_flows("2023-07-09", "2023-07-10", resolution="native")
peak_row = peak(hourly_border_counts(ci_day, "Counter-intuitive"))
print(f"Peak: {peak_row['n_borders']} borders counter-intuitive at {peak_row['datetime']}")

fig = border_count_price_chart(
    ci_day,
    "Counter-intuitive",
    "Counter-intuitive borders vs. average day-ahead price, 2023-07-09 (UTC)",
    peak_vline="2023-07-09 09:00:00",
    peak_label="Bertelli's reported peak",
)
save_chart(fig, "ci-peak-2023-07-09")
fig.show()

## The counter-intuitive flow leaderboard

Since CI flows have become extremely common, we wondered what would be the best way of sizing them up.

And while a single day's single hour is interesting, at the end of the day, it is just a snapshot.

A full month, however, is a unit worth ranking on.

`flow_type_summary` gives the same leaderboard metrics for any range &ndash; hours flagged, and the implied value of the price gap while flagged. Note the second is the value the spread is capturing, not a welfare or economic-cost figure (see the function's docstring). Feel free to play around with that range.

In [ ]:
ci_month = fetch_flows_range("2023-07-01", "2023-08-01")
ci_leaderboard = flow_type_summary(ci_month, "Counter-intuitive")

for metric, label in [
    ("n_hours", "Hours flagged counter-intuitive"),
    ("implied_value_eur", "Implied value of the price gap (EUR)"),
]:
    fig = top_n_bar(
        ci_leaderboard,
        metric,
        label,
        f"Top 10 borders by {label}, 2023-07",
        FLOW_COLORS["Counter-intuitive"],
    )
    save_chart(fig, f"ci-leaderboard-2023-07-{metric}")
    fig.show()

Bertelli's article walks through why one specific counter-intuitive pattern made sense despite looking wrong: at 11:00 CET on 2023-07-09, Hungary re-exported Romanian power to Austria, Croatia, Slovenia, and Slovakia at a per-border loss, because the HU-RO price spread was so large that importing from Romania and re-exporting elsewhere was still a net win overall.

The leaderboard above shows where this pattern is most common; the cell below reproduces his specific arithmetic directly from the same peak-hour data &ndash; no manual lookup, same formula (`|net_flow_mw| x |price_delta|`) applied per border.

<img src="media/hu-re-export-2023-07-09-dashboard.png" alt="VoltPath dashboard filtered to Hungary's borders on 2023-07-09, showing the re-export pattern" width="800">

In [ ]:
peak_hour = ci_day[ci_day["datetime"] == peak_row["datetime"]]

# The borders Bertelli's worked example covers: HU re-exporting Romanian power onward.
re_export_borders = ["AT -> HU", "HR -> HU", "HU -> SI", "HU -> SK"]
import_border = "HU -> RO"

re_exports = peak_hour[peak_hour["border"].isin(re_export_borders)].copy()
re_exports["value_eur"] = -(re_exports["net_flow_mw"].abs() * re_exports["price_delta"].abs()).round(0)
re_exports["leg"] = "Re-export loss"

import_leg = peak_hour[peak_hour["border"] == import_border].iloc[0]
import_row = pd.DataFrame(
    [
        {
            "border": import_border,
            "value_eur": round(abs(import_leg["net_flow_mw"]) * abs(import_leg["price_delta"]), 0),
            "leg": "Import gain",
        }
    ]
)

legs = pd.concat([import_row, re_exports[["border", "value_eur", "leg"]]], ignore_index=True)
net_effect = legs["value_eur"].sum()

display(legs.style.format({"value_eur": "EUR {:,.0f}"}).hide(axis="index"))
print(f"Net value effect: EUR {net_effect:,.0f}")

legs_sorted = legs.sort_values("value_eur")
net_row = pd.DataFrame([{"border": "Net value effect", "value_eur": net_effect, "leg": "Net effect"}])
plot_df = pd.concat([net_row, legs_sorted], ignore_index=True)

fig = px.bar(
    plot_df,
    x="value_eur",
    y="border",
    color="leg",
    orientation="h",
    title=f"HU re-export economics, peak counter-intuitive hour ({peak_row['datetime']:%Y-%m-%d %H:%M} UTC)",
    labels={"value_eur": "Value (EUR)", "border": "Border", "leg": ""},
    color_discrete_map={
        "Import gain": FLOW_COLORS["Convergence"],  # matches volumeImport on the VoltPath dashboard
        "Re-export loss": FLOW_COLORS["Congestion"],  # matches volumeExport
        "Net effect": FLOW_COLORS["Counter-intuitive"],  # matches volumeNet (neutral slate)
    },
)
fig.update_yaxes(categoryorder="array", categoryarray=plot_df["border"].tolist())
fig.add_vline(x=0, line_color="black")
fig.add_hline(y=0.5, line_color="black", line_width=1)
save_chart(fig, "hu-re-export-2023-07-09")
fig.show()

## Notes

- Small differences from Bertelli's published numbers are expected: a manual analysis and an automated one will treat edge cases slightly differently. That's a feature of having a live, queryable platform alongside his original write-up, not a correction of it.
- This work led us to a congestion cost/leaderboard analysis which has its own notebook: [congestion-analysis](../congestion-analysis/congestion_analysis.ipynb).

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).